In [2]:
#:: 1. Verify your high-performance Torch + CUDA 12.4 baseline is fully locked
%pip install torch==2.6.0+cu124 torchvision==0.21.0+cu124 torchaudio==2.6.0+cu124 --index-url https://download.pytorch.org/whl/cu124

#:: 2. Install matching xFormers to manage VRAM compression
%pip install xformers==0.0.29.post3 --index-url https://download.pytorch.org/whl/cu124

#:: 3. Install core Hugging Face Diffusion engine layers
%pip install diffusers==0.32.2 transformers==4.48.2 accelerate==1.3.0 peft==0.14.0

#:: 4. Install memory optimization backends (Essential for 6GB VRAM limits)
%pip install bitsandbytes==0.45.1

#:: 5. Install standard image & utility layers
%pip install Pillow numpy tqdm

Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
Note: you may need to restart the kernel to use updated packages.
  Using cached diffusers-0.32.2-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-4.48.2-py3-none-any.whl.metadata (44 kB)
  Using cached accelerate-1.3.0-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
Using cached diffusers-0.32.2-py3-none-any.whl (3.2 MB)
Using cached transformers-4.48.2-py3-none-any.whl (9.7 MB)
Using cached accelerate-1.3.0-py3-none-any.whl (336 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-win_amd64.whl (2.5 MB)

  Attempting uninstall: huggingface-hub

    Found existing installation: huggingface_hub 1.15.0

    Un

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 4.44.1 requires aiofiles<24.0,>=22.0, but you have aiofiles 24.1.0 which is incompatible.
gradio 4.44.1 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
gradio-client 1.3.0 requires websockets<13.0,>=10.0, but you have websockets 16.0 which is incompatible.
ragas 0.4.3 requires datasets>=4.0.0, but you have datasets 3.2.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import sys
import os

print("--- PYTHON ENVIRONMENT INFO ---")
print("Executable path:", sys.executable)
print("Python Version:", sys.version)
print("\n--- TRYING IMPORTS ---")

packages = ["torch", "diffusers", "transformers", "peft"]
for pkg in packages:
    try:
        mod = __import__(pkg)
        print(f"✅ {pkg}: Imported successfully from {os.path.dirname(mod.__file__)}")
    except ImportError as e:
        print(f"❌ {pkg}: Failed to import -> {e}")

--- PYTHON ENVIRONMENT INFO ---
Executable path: c:\Users\sures\AppData\Local\Programs\Python\Python311\python.exe
Python Version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]

--- TRYING IMPORTS ---
✅ torch: Imported successfully from c:\Users\sures\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch
✅ diffusers: Imported successfully from c:\Users\sures\AppData\Local\Programs\Python\Python311\Lib\site-packages\diffusers
✅ transformers: Imported successfully from c:\Users\sures\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers
✅ peft: Imported successfully from c:\Users\sures\AppData\Local\Programs\Python\Python311\Lib\site-packages\peft


In [ ]:
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained("stable-diffusion-v1-5/stable-diffusion-v1-5", dtype=torch.bfloat16, device_map="cuda")

prompt = "Astronaut in a jungle, cold color palette, muted colors, detailed, 8k"
image = pipe(prompt).images[0]

In [14]:
import os
import sys
import torch

# Optimal environment VRAM constraint layer for your 6GB RTX 4050
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    import xformers
    import xformers.ops
    XFORMERS_AVAILABLE = True
except Exception as e:
    XFORMERS_AVAILABLE = False
    print(f"⚠️ xformers failed to load, falling back to standard attention: {e}")

# Use inline type ignore tags to suppress IDE-specific lazy-loading false positives
from diffusers import DiffusionPipeline, AutoencoderKL, DDPMScheduler  # type: ignore
from diffusers.utils import load_image  # type: ignore
from accelerate import Accelerator

print(f"--- Verification ---")
print(f"✅ Python Version: {sys.version.split()[0]}")
print(f"✅ PyTorch Build: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ Active GPU Asset: {torch.cuda.get_device_name(0)}")
print(f"✅ xformers Memory Optimization: {XFORMERS_AVAILABLE}")


--- Verification ---
✅ Python Version: 3.11.9
✅ PyTorch Build: 2.6.0+cu124
✅ CUDA Available: True
✅ Active GPU Asset: NVIDIA GeForce RTX 4050 Laptop GPU
✅ xformers Memory Optimization: True


In [16]:
import os
import warnings

# 1. Block the harmless xformers/Triton traceback printout
os.environ["XFORMERS_FORCE_DISABLE_TRITON"] = "1"

# 2. Silence the Jupyter tqdm widget warning
warnings.filterwarnings("ignore", category=UserWarning, module="tqdm")

In [ ]:
import sys, site, os, platform
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Prefix:", sys.prefix)
print("Base prefix:", sys.base_prefix)
print("Platform:", platform.platform())
print("User site:", site.getusersitepackages())
print("Site packages:", site.getsitepackages() if hasattr(site, "getsitepackages") else "N/A")
packages = [
    "torch",
    "torchvision",
    "torchaudio",
    "diffusers",
    "transformers",
    "accelerate",
    "peft",
    "safetensors",
    "xformers",
    "streamlit",
    "Pillow",
    "numpy",
]
for pkg in packages:
    try:
        mod = __import__(pkg if pkg != "Pillow" else "PIL")
        v = getattr(mod, "__version__", "no __version__")
        print(f"{pkg}: {v}")
    except Exception as e:
        print(f"{pkg}: not importable -> {e}")
        
print(sys.executable)
print(sys.prefix)
print(sys.base_prefix)

Python executable: c:\Users\sures\AppData\Local\Programs\Python\Python311\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Prefix: c:\Users\sures\AppData\Local\Programs\Python\Python311
Base prefix: c:\Users\sures\AppData\Local\Programs\Python\Python311
Platform: Windows-10-10.0.26200-SP0
User site: C:\Users\sures\AppData\Roaming\Python\Python311\site-packages
Site packages: ['c:\\Users\\sures\\AppData\\Local\\Programs\\Python\\Python311', 'c:\\Users\\sures\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages']
torch: 2.6.0+cu124
torchvision: 0.21.0+cu124
torchaudio: 2.6.0+cu124
diffusers: 0.31.0
transformers: 4.48.2
accelerate: 1.3.0
peft: 0.14.0
safetensors: 0.8.0-rc.0
xformers: 0.0.29.post3
streamlit: 1.57.0
Pillow: 10.4.0
numpy: 2.4.4
c:\Users\sures\AppData\Local\Programs\Python\Python311\python.exe
c:\Users\sures\AppData\Local\Programs\Python\Python311
c:\Users\sures\AppData\Local\Programs\Python\Python311
